<a href="https://colab.research.google.com/github/ADITYA-dev02/Data-science-/blob/main/international.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## <b>Analysing the csv file </b>

<br>





### **Analyse 1. Rename all the column names to their appropriate names, for example meta.created should be renamed as created_date**




In [2]:
import pandas as pd

df = pd.read_csv("International_T20_Data.csv")

df = df.rename(columns={ "meta.created": "created_date", "meta.data_version": "data_version", "meta.revision": "revision", "info.city": "city", "info.venue": "venue", "info.match_type": "match_type", "info.overs": "overs", "info.teams": "teams", "info.toss.winner": "toss_winner", "info.toss.decision": "toss_decision", "info.player_of_match": "player_of_match", "info.outcome.winner": "match_winner" })

print("Renamed Columns:",df.columns)

Renamed Columns: Index(['innings', 'data_version', 'created_date', 'revision', 'info.dates',
       'info.gender', 'match_type', 'info.outcome.by.wickets', 'match_winner',
       'overs', 'player_of_match', 'teams', 'toss_decision', 'toss_winner',
       'info.umpires', 'venue', 'city', 'info.outcome.by.runs',
       'info.match_type_number', 'info.neutral_venue', 'info.outcome.method',
       'info.outcome.result', 'info.outcome.eliminator',
       'info.supersubs.New Zealand', 'info.supersubs.South Africa',
       'info.bowl_out', 'info.outcome.bowl_out'],
      dtype='object')


## **Analyse 2. Find out the top three venues which hosted the greatest number of matches.**

In [3]:
import pandas as pd

df = pd.read_csv("International_T20_Data.csv")

df = df.rename(columns={"info.venue": "venue"})

top_venues = df["venue"].value_counts().head(3)

print("Top 3 venues with most matches:")
print(top_venues)

Top 3 venues with most matches:
venue
Dubai International Cricket Stadium    62
Sheikh Zayed Stadium                   41
Shere Bangla National Stadium          39
Name: count, dtype: int64


## **Analyse 3. Find out the pair of cricket teams who played the most number of T20 matches against each other.**



In [4]:
import pandas as pd
import ast

df = pd.read_csv("International_T20_Data.csv")

df["teams"] = df["info.teams"].apply(ast.literal_eval)


df["pair"] = df["teams"].apply(lambda x: tuple(sorted(x)))


pair_count = df["pair"].value_counts()

print("Pair of teams who played most matches:")
print(pair_count.head(1))

Pair of teams who played most matches:
pair
(Australia, England)    45
Name: count, dtype: int64


## **Analyse 4. Print the top five teams by their win percentages. Win percentage is defined as the number of matches won divided by the number of matches played and then multiplied by 100.**



In [7]:
import pandas as pd
import ast

df = pd.read_csv("International_T20_Data.csv")

df["teams"] = df["info.teams"].apply(ast.literal_eval)

df["team1"] = df["teams"].apply(lambda x: x[0])
df["team2"] = df["teams"].apply(lambda x: x[1])

matches_played = pd.concat([df["team1"], df["team2"]]).value_counts()


matches_won = df["info.outcome.winner"].value_counts()

team_stats = pd.DataFrame({
    "played": matches_played,
    "won": matches_won
}).fillna(0)


team_stats["win_percentage"] = (team_stats["won"] / team_stats["played"]) * 100

top5 = team_stats.sort_values(by="win_percentage", ascending=False).head(5)

print("Top 5 teams by win percentage:")
print(top5)

Top 5 teams by win percentage:
             played   won  win_percentage
Belgium           3   3.0      100.000000
Spain             6   5.0       83.333333
Germany          17  13.0       76.470588
Namibia          34  25.0       73.529412
Afghanistan      75  51.0       68.000000


## **Analyse 5. Write a function to get the scorecard of each match. This function would take the innings value as argument and return two scorecard dataframes each for one team as shown below. So the first dataframe would contain the top 4 scorers of the team who batted first and the top 4 bowlers of the opponent team. And the second dataframe would contain the top 4 scorers of the team who batted second and the top 4 bowlers of the opponent team.**



In [8]:
import pandas as pd
import ast

df = pd.read_csv("International_T20_Data.csv")

innings = ast.literal_eval(df.loc[0, "innings"])

def get_scorecard(innings):

    first_innings = innings[0]["1st innings"]["deliveries"]
    second_innings = innings[1]["2nd innings"]["deliveries"]

    first_list = []
    for ball in first_innings:
        for k, v in ball.items():
            first_list.append(v)

    second_list = []
    for ball in second_innings:
        for k, v in ball.items():
            second_list.append(v)

    df1 = pd.json_normalize(first_list)
    df2 = pd.json_normalize(second_list)

    top_bat1 = df1.groupby("batsman")["runs.batsman"].sum().sort_values(ascending=False).head(4)
    top_bat2 = df2.groupby("batsman")["runs.batsman"].sum().sort_values(ascending=False).head(4)

    top_bowl2 = df1.groupby("bowler")["runs.total"].sum().sort_values().head(4)
    top_bowl1 = df2.groupby("bowler")["runs.total"].sum().sort_values().head(4)

    scorecard1 = pd.concat([top_bat1, top_bowl2], axis=1)
    scorecard2 = pd.concat([top_bat2, top_bowl1], axis=1)

    return scorecard1, scorecard2


score1, score2 = get_scorecard(innings)

print("Scorecard First Innings")
print(score1)

print("\nScorecard Second Innings")
print(score2)

Scorecard First Innings
                runs.batsman  runs.total
AJ Finch                43.0         NaN
M Klinger               38.0         NaN
TM Head                 31.0         NaN
AJ Turner               18.0         NaN
DAS Gunaratne            NaN        11.0
S Prasanna               NaN        24.0
SL Malinga               NaN        29.0
PADLR Sandakan           NaN        31.0

Scorecard Second Innings
                 runs.batsman  runs.total
DAS Gunaratne            52.0         NaN
EMDY Munaweera           44.0         NaN
N Dickwella              30.0         NaN
TAM Siriwardana          15.0         NaN
AJ Turner                 NaN        12.0
A Zampa                   NaN        26.0
JP Faulkner               NaN        29.0
PJ Cummins                NaN        30.0
